# ETLegacy Core Infrastructure

## Author: Zach Etienne

To run the whole notebook, click the `>>` toolbar button and choose
**Restart Kernel and Run All Cells...**.

This notebook builds a small ETLegacy thorn from NRPy registrations, writes the
generated CCL metadata and source files, inspects the complete generated
artifacts, and validates that the thorn contains the expected variables,
parameters, schedule entries, build metadata, and right-hand-side source.

**Notebook Status:** Validated for generated ETLegacy thorn content. External
Einstein Toolkit build/run not executed by this notebook.

**Validation Notes:** The validation section checks the generated thorn
inventory, CCL metadata, schedule entries, build metadata, and complete RHS
source for `ToyETLegacyNRPy`. A full runtime test requires an Einstein Toolkit
checkout.

Navigation:
[Index](../index.ipynb) |
Previous: [BHaH Project Anatomy](bhah_project_anatomy.ipynb) |
Next: [Carpet WaveToy Thorns](carpet_wavetoy_thorns.ipynb)

### Required Reading

- [C Function Registration](../1-intro/c_function.ipynb): introduces the NRPy
  CFunction registry used here.
- [BHaH Project Anatomy](bhah_project_anatomy.ipynb): introduces generated
  project vocabulary used again for ETLegacy thorns.

### NRPy Source Code

NRPy is used here as a pip-installed package. The import paths below identify
the installed modules used by the notebook; [NRPy upstream source][nrpy-source]
is the source browser for those modules.

- `nrpy.c_function`: implements the CFunction registry used by the RHS
  registration.
- `nrpy.infrastructures.ETLegacy.interface_ccl`: writes variable and inherited
  capability declarations.
- `nrpy.infrastructures.ETLegacy.param_ccl`: writes runtime parameter
  declarations.
- `nrpy.infrastructures.ETLegacy.schedule_ccl`: writes schedule metadata.
- `nrpy.infrastructures.ETLegacy.make_code_defn`: writes the ETLegacy
  source-file list.
- `nrpy.infrastructures.ETLegacy.simple_loop`: emits the ETLegacy grid loop
  used in the RHS.

[nrpy-source]: https://github.com/nrpy/nrpy

# Table of Contents

1. [Words for This Notebook](#Words-for-This-Notebook)
1. [Workspace and Generated Files](#Workspace-and-Generated-Files)
1. [Step 1: Prepare the Thorn Workspace](#Step-1:-Prepare-the-Thorn-Workspace)
1. [Step 2: Register Gridfunctions and CodeParameters][step2]
1. [Step 3: Build the RHS C Function Body](#Step-3:-Build-the-RHS-C-Function-Body)
1. [Step 4: Register the RHS C Function](#Step-4:-Register-the-RHS-C-Function)
1. [Step 5: Register Workflow Hooks](#Step-5:-Register-Workflow-Hooks)
1. [Step 6: Write ETLegacy Thorn Files](#Step-6:-Write-ETLegacy-Thorn-Files)
1. [Step 7: Catalog the Generated Thorn](#Step-7:-Catalog-the-Generated-Thorn)
1. [Step 8: Inspect CCL Files](#Step-8:-Inspect-CCL-Files)
1. [Step 9: Inspect Build Metadata and Source](#Step-9:-Inspect-Build-Metadata-and-Source)
1. [Validation Check](#Validation-Check)
1. [What Next?](#What-Next?)

[step2]: #Step-2:-Register-Gridfunctions-and-CodeParameters

# Words for This Notebook
### [Back to [top](#Table-of-Contents)]

- **Einstein Toolkit:** a framework that builds and runs components called
  thorns.
- **ETLegacy:** NRPy's writer for traditional Einstein Toolkit CCL metadata and
  C source files.
- **Thorn:** a component directory containing CCL metadata and source files.
- **CCL:** Cactus Configuration Language, the metadata language used by
  Einstein Toolkit thorns.
- **`interface.ccl`:** the CCL file that declares variables and inherited
  capabilities.
- **`param.ccl`:** the CCL file that declares runtime parameters.
- **`schedule.ccl`:** the CCL file that says when generated functions run.
- **`make.code.defn`:** the source-file list consumed by the Toolkit build.
- **Gridfunction:** a field stored on the grid; here `psi`, `psi_rhs`, and
  `energy` become ETLegacy gridfunction names.
- **CodeParameter:** an NRPy parameter that becomes generated CCL or C code.
- **C function:** a generated C routine registered with NRPy.
- **Prototype:** the C declaration of a generated function.
- **Parameter list:** the formal arguments of a C function; ETLegacy scheduled
  functions use `CCTK_ARGUMENTS`.
- **Body:** the C statements inside the generated function.
- **Schedule:** the ordered runtime workflow described by `schedule.ccl`.
- **Schedule bin:** a named phase in that workflow, such as `MoL_CalcRHS`.
- **Method of Lines:** the time-integration pattern that calls RHS functions
  from the `MoL_CalcRHS` schedule bin.

# Workspace and Generated Files
### [Back to [top](#Table-of-Contents)]

This notebook owns and recreates
`project/etlegacy_core_infrastructure/` under the `5-infrastructures`
directory. Each run removes only that owned directory and writes one generated
thorn at:

`project/etlegacy_core_infrastructure/toy_etlegacy/ToyETLegacyNRPy`

The key generated artifacts are:

| Artifact | Purpose | Where used | What to inspect |
| --- | --- | --- | --- |
| `interface.ccl` | Declares gridfunction groups | ET thorn metadata | groups |
| `param.ccl` | Declares runtime parameters | ET parameter parser | parameters |
| `schedule.ccl` | Schedules generated functions | runtime workflow | bins |
| `src/make.code.defn` | Lists compiled source files | build metadata | files |
| `src/ToyETLegacyNRPy_rhs_eval.c` | Evaluates the toy RHS | RHS function | loop |

# Step 1: Prepare the Thorn Workspace
### [Back to [top](#Table-of-Contents)]

This setup cell defines a stable generated-project location and small helper
functions. Skim the helpers on a first pass; their purpose is path stability,
inventory printing, and validation checks.

In [1]:
from pathlib import Path
import contextlib
import io
import re
import shutil


def find_notebook_dir():
    cwd = Path.cwd().resolve()
    if cwd.name == "5-infrastructures" and (cwd / "backend_choice_guide.ipynb").is_file():
        return cwd
    candidate = cwd / "5-infrastructures"
    if (candidate / "backend_choice_guide.ipynb").is_file():
        return candidate.resolve()
    raise RuntimeError("Run this notebook from the repository root or 5-infrastructures/.")


NOTEBOOK_DIR = find_notebook_dir()

These helpers print file inventories and validate required substrings in
generated files. Validation failures are reported as `RuntimeError`s.

In [2]:
def file_inventory(root):
    return sorted(str(path.relative_to(root)) for path in root.rglob("*") if path.is_file())


def check_contains(path, required_substrings):
    text = path.read_text(encoding="utf-8", errors="replace")
    missing = [item for item in required_substrings if item not in text]
    return text, missing


def print_check(name, passed, detail):
    status = "PASS" if passed else "FAIL"
    print(f"{status:4} | {name} | {detail}")

Now create the owned workspace and print the path before any generated files
are written.

In [3]:
WORKSPACE = NOTEBOOK_DIR / "project" / "etlegacy_core_infrastructure"
PROJECT_DIR = WORKSPACE / "toy_etlegacy"
THORN_NAME = "ToyETLegacyNRPy"
THORN_DIR = PROJECT_DIR / THORN_NAME

if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir(parents=True)

print("workspace:", WORKSPACE.relative_to(NOTEBOOK_DIR))
print("project path:", PROJECT_DIR.relative_to(NOTEBOOK_DIR))
print("thorn path:", THORN_DIR.relative_to(NOTEBOOK_DIR))

workspace: project/etlegacy_core_infrastructure
project path: project/etlegacy_core_infrastructure/toy_etlegacy
thorn path: project/etlegacy_core_infrastructure/toy_etlegacy/ToyETLegacyNRPy


# Step 2: Register Gridfunctions and CodeParameters
### [Back to [top](#Table-of-Contents)]

The toy thorn uses one evolved field, `psi`, one RHS field, `psi_rhs`, and one
auxiliary field, `energy`. The two CodeParameters, `fd_order` and
`wave_speed`, become generated metadata or C reads.

In [4]:
import nrpy.c_function as cfc
import nrpy.grid as gri
import nrpy.params as par
from nrpy.infrastructures.ETLegacy import (
    CodeParameters,
    MoL_registration,
    Symmetry_registration,
    boundary_conditions,
    interface_ccl,
    make_code_defn,
    param_ccl,
    schedule_ccl,
    simple_loop,
    zero_rhss,
)

par.set_parval_from_str("Infrastructure", "ETLegacy")

Before registering tutorial-owned objects, clear only the names this notebook
owns. This keeps rerunning the notebook from the top deterministic.

In [5]:
for name in [
    f"{THORN_NAME}_rhs_eval",
    f"{THORN_NAME}_zero_rhss",
    f"{THORN_NAME}_MoL_registration",
    f"{THORN_NAME}_Symmetry_registration_oldCartGrid3D",
    f"{THORN_NAME}_specify_Driver_BoundaryConditions",
]:
    cfc.CFunction_dict.pop(name, None)
for name in ["psi", "energy", "psi_rhs", "energy_rhs"]:
    gri.glb_gridfcs_dict.pop(name, None)
for name in ["fd_order", "wave_speed"]:
    par.glb_code_params_dict.pop(name, None)

Register the gridfunctions and parameters, then print the learner-facing
catalog of names and roles.

In [6]:
par.register_CodeParameter("CCTK_INT", __name__, "fd_order", 4)
par.register_CodeParameter("CCTK_REAL", __name__, "wave_speed", 1.0)
evolved_fields = gri.register_gridfunctions(["psi"], group="EVOL")
aux_fields = gri.register_gridfunctions(["energy"], group="AUX")

print("gridfunction catalog:")
print("EVOL:", [str(field) for field in evolved_fields])
print("AUX:", [str(field) for field in aux_fields])
print("CodeParameters:", ["fd_order", "wave_speed"])

gridfunction catalog:
EVOL: ['psi']
AUX: ['energy']
CodeParameters: ['fd_order', 'wave_speed']


`psi` and `energy` now have ETLegacy gridfunction names. The CodeParameter
names will appear in generated CCL and source files.

# Step 3: Build the RHS C Function Body
### [Back to [top](#Table-of-Contents)]

The desired C function shape is schematic here; placeholders mark where the
generated loop and memory accesses will appear. This is not validation
evidence. The complete generated source is printed in Step 9.

```c
void ToyETLegacyNRPy_rhs_eval(CCTK_ARGUMENTS) {
    DECLARE_CCTK_ARGUMENTS_ToyETLegacyNRPy_rhs_eval;
    DECLARE_CCTK_PARAMETERS;
    const CCTK_REAL wave_speed = ...;
    for (...) {
        psi_rhs[...] = wave_speed * (energy[...] - psi[...]);
    }
}
```

The body first declares the scheduled function arguments and parameters, then
reads `wave_speed`, and finally emits an ETLegacy loop over interior grid
points.

In [7]:
body = f"""DECLARE_CCTK_ARGUMENTS_{THORN_NAME}_rhs_eval;
DECLARE_CCTK_PARAMETERS;
"""
body += CodeParameters.read_CodeParameters(
    list_of_tuples__thorn_CodeParameter=[(THORN_NAME, "wave_speed")],
    declare_invdxxs=True,
)
body += simple_loop.simple_loop(
    loop_body=(
        f"{gri.ETLegacyGridFunction.access_gf('psi_rhs')} = "
        f"wave_speed * ({gri.ETLegacyGridFunction.access_gf('energy')} - "
        f"{gri.ETLegacyGridFunction.access_gf('psi')});"
    ),
    loop_region="interior",
)

print(body)

DECLARE_CCTK_ARGUMENTS_ToyETLegacyNRPy_rhs_eval;
DECLARE_CCTK_PARAMETERS;
const CCTK_REAL *wave_speedptr = CCTK_ParameterGet("wave_speed", "ToyETLegacyNRPy", NULL);  // ToyETLegacyNRPy::wave_speed
const CCTK_REAL wave_speed = *wave_speedptr;
const CCTK_REAL invdxx0 = 1.0/CCTK_DELTA_SPACE(0);
const CCTK_REAL invdxx1 = 1.0/CCTK_DELTA_SPACE(1);
const CCTK_REAL invdxx2 = 1.0/CCTK_DELTA_SPACE(2);
#pragma omp parallel for
for (int i2 = cctk_nghostzones[2]; i2 < cctk_lsh[2]-cctk_nghostzones[2]; i2++) {
for (int i1 = cctk_nghostzones[1]; i1 < cctk_lsh[1]-cctk_nghostzones[1]; i1++) {
for (int i0 = cctk_nghostzones[0]; i0 < cctk_lsh[0]-cctk_nghostzones[0]; i0++) {
psi_rhsGF[CCTK_GFINDEX3D(cctkGH, i0, i1, i2)] = wave_speed * (energyGF[CCTK_GFINDEX3D(cctkGH, i0, i1, i2)] - psiGF[CCTK_GFINDEX3D(cctkGH, i0, i1, i2)]);
} // END LOOP: for i0 over [cctk_nghostzones[0], cctk_lsh[0]-cctk_nghostzones[0])
} // END LOOP: for i1 over [cctk_nghostzones[1], cctk_lsh[1]-cctk_nghostzones[1])
} // END LOOP: for i

The body includes the scheduled-function argument declaration, the parameter
read for `wave_speed`, and the assignment to `psi_rhs`.

# Step 4: Register the RHS C Function
### [Back to [top](#Table-of-Contents)]

The table maps the desired C function pieces to NRPy registration fields.

| C function piece | NRPy registration field |
| --- | --- |
| description comment | `desc` |
| return type | `cfunc_type` |
| function name | `name` |
| parameter list | `params` |
| includes | `includes` |
| body | `body` |
| output thorn subdirectory | `subdirectory` |
| schedule metadata | `ET_schedule_bins_entries` |

In [8]:
includes = ["cctk.h", "cctk_Arguments.h", "cctk_Parameters.h"]
desc = "Evaluate a toy right-hand side."
cfunc_type = "void"
name = f"{THORN_NAME}_rhs_eval"
params = "CCTK_ARGUMENTS"
schedule_bin = "MoL_CalcRHS"
schedule_entry = """
schedule FUNC_NAME in MoL_CalcRHS
{
  LANG: C
  READS: evol_variables(everywhere)
  READS: aux_variables(everywhere)
  WRITES: evol_variables_rhs(interior)
} "Evaluate toy right-hand side"
"""

Now register the function. The compact summary is useful metadata, but the
complete generated source printed later is the required evidence.

In [9]:
cfc.register_CFunction(
    subdirectory=THORN_NAME,
    includes=includes,
    desc=desc,
    cfunc_type=cfunc_type,
    name=name,
    params=params,
    body=body,
    ET_thorn_name=THORN_NAME,
    ET_schedule_bins_entries=[(schedule_bin, schedule_entry)],
    ET_current_thorn_CodeParams_used=["wave_speed"],
)
registered = cfc.CFunction_dict[name]
print("registered function:", name)
print("prototype:", registered.function_prototype)
print("includes:", includes)
print("schedule bin:", schedule_bin)
print("CodeParameters used:", ["wave_speed"])

registered function: ToyETLegacyNRPy_rhs_eval
prototype: void ToyETLegacyNRPy_rhs_eval(CCTK_ARGUMENTS);
includes: ['cctk.h', 'cctk_Arguments.h', 'cctk_Parameters.h']
schedule bin: MoL_CalcRHS
CodeParameters used: ['wave_speed']


The registry now contains the RHS function and its ETLegacy schedule metadata.

# Step 5: Register Workflow Hooks
### [Back to [top](#Table-of-Contents)]

ETLegacy thorns also need hook functions for zeroing RHSs, Method-of-Lines
registration, symmetry registration, and boundary-condition registration.

In [10]:
zero_rhss.register_CFunction_zero_rhss(THORN_NAME)
MoL_registration.register_CFunction_MoL_registration(THORN_NAME)
Symmetry_registration.register_CFunction_Symmetry_registration_oldCartGrid3D(THORN_NAME)
boundary_conditions.register_CFunction_specify_Driver_BoundaryConditions(
    thorn_name=THORN_NAME
)

print("registered C functions:")
for function_name in sorted(name for name in cfc.CFunction_dict if name.startswith(THORN_NAME)):
    print(function_name)

registered C functions:
ToyETLegacyNRPy_MoL_registration
ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D
ToyETLegacyNRPy_rhs_eval
ToyETLegacyNRPy_specify_Driver_BoundaryConditions
ToyETLegacyNRPy_zero_rhss


The registered hook names will become generated C source files and schedule
entries in the thorn.

# Step 6: Write ETLegacy Thorn Files
### [Back to [top](#Table-of-Contents)]

The ETLegacy writers create the CCL files and output each registered C
function into the thorn source directory.

In [11]:
generation_output = io.StringIO()
uses_includes = "USES INCLUDE: Symmetry.h\nUSES INCLUDE: Boundary.h\n"
storage = """
STORAGE: evol_variables[3]
STORAGE: evol_variables_rhs[1]
STORAGE: aux_variables[1]
"""

with contextlib.redirect_stdout(generation_output):
    interface_ccl.construct_interface_ccl(
        project_dir=str(PROJECT_DIR),
        thorn_name=THORN_NAME,
        inherits="Boundary Grid MethodofLines",
        USES_INCLUDEs=uses_includes,
        is_evol_thorn=True,
    )
    param_ccl.construct_param_ccl(project_dir=str(PROJECT_DIR), thorn_name=THORN_NAME)
    schedule_ccl.construct_schedule_ccl(
        project_dir=str(PROJECT_DIR), thorn_name=THORN_NAME, STORAGE=storage
    )
    make_code_defn.output_CFunctions_and_construct_make_code_defn(
        project_dir=str(PROJECT_DIR), thorn_name=THORN_NAME
    )

cleaned_generation_output = re.sub(
    r"\x1b\[[0-?]*[ -/]*[@-~]", "", generation_output.getvalue()
)
cleaned_generation_output = cleaned_generation_output.replace(str(NOTEBOOK_DIR), ".")
print(cleaned_generation_output.strip())

Checking ./project/etlegacy_core_infrastructure/toy_etlegacy/ToyETLegacyNRPy/interface.ccl...[written]
Checking ./project/etlegacy_core_infrastructure/toy_etlegacy/ToyETLegacyNRPy/param.ccl...[written]
Checking ./project/etlegacy_core_infrastructure/toy_etlegacy/ToyETLegacyNRPy/schedule.ccl...[written]
Checking ./project/etlegacy_core_infrastructure/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_MoL_registration.c...[written]
Checking ./project/etlegacy_core_infrastructure/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D.c...[written]
Checking ./project/etlegacy_core_infrastructure/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_rhs_eval.c...[written]
Checking ./project/etlegacy_core_infrastructure/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_specify_Driver_BoundaryConditions.c...[written]
Checking ./project/etlegacy_core_infrastructure/toy_etlegacy/ToyETLegacyNRPy/src/ToyETLegacyNRPy_zero_rhss.c...[written]
Checking ./project/etlegacy_core_in

The generator messages are secondary evidence. The complete inventory and
files below are the required generated artifacts.

# Step 7: Catalog the Generated Thorn
### [Back to [top](#Table-of-Contents)]

This inventory must exactly match the files expected for this toy thorn.

In [12]:
EXPECTED_THORN_INVENTORY = [
    "ToyETLegacyNRPy/interface.ccl",
    "ToyETLegacyNRPy/param.ccl",
    "ToyETLegacyNRPy/schedule.ccl",
    "ToyETLegacyNRPy/src/ToyETLegacyNRPy_MoL_registration.c",
    "ToyETLegacyNRPy/src/ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D.c",
    "ToyETLegacyNRPy/src/ToyETLegacyNRPy_rhs_eval.c",
    "ToyETLegacyNRPy/src/ToyETLegacyNRPy_specify_Driver_BoundaryConditions.c",
    "ToyETLegacyNRPy/src/ToyETLegacyNRPy_zero_rhss.c",
    "ToyETLegacyNRPy/src/make.code.defn",
]

thorn_inventory = file_inventory(PROJECT_DIR)
for relative_path in thorn_inventory:
    print(relative_path)

ToyETLegacyNRPy/interface.ccl
ToyETLegacyNRPy/param.ccl
ToyETLegacyNRPy/schedule.ccl
ToyETLegacyNRPy/src/ToyETLegacyNRPy_MoL_registration.c
ToyETLegacyNRPy/src/ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D.c
ToyETLegacyNRPy/src/ToyETLegacyNRPy_rhs_eval.c
ToyETLegacyNRPy/src/ToyETLegacyNRPy_specify_Driver_BoundaryConditions.c
ToyETLegacyNRPy/src/ToyETLegacyNRPy_zero_rhss.c
ToyETLegacyNRPy/src/make.code.defn


The inventory shows one thorn directory, three CCL files, four generated hook
sources, the RHS source, and the build metadata file.

# Step 8: Inspect CCL Files
### [Back to [top](#Table-of-Contents)]

The full CCL files are printed because CCL metadata is the lesson's main
generated artifact. Inspect variable groups in `interface.ccl`, runtime
parameters in `param.ccl`, and schedule bins in `schedule.ccl`.

In [13]:
interface_text = (THORN_DIR / "interface.ccl").read_text(encoding="utf-8", errors="replace")
param_text = (THORN_DIR / "param.ccl").read_text(encoding="utf-8", errors="replace")
schedule_text = (THORN_DIR / "schedule.ccl").read_text(encoding="utf-8", errors="replace")

for relative_path, ccl_text in [
    ("interface.ccl", interface_text),
    ("param.ccl", param_text),
    ("schedule.ccl", schedule_text),
]:
    print(f"--- {relative_path} ---")
    print(ccl_text)

--- interface.ccl ---

# This interface.ccl file was automatically generated by NRPy.
#   You are advised against modifying it directly; instead
#   modify the Python code that generates it.

# With "implements", we give our thorn its unique name.
implements: ToyETLegacyNRPy

# By "inheriting" other thorns, we tell the Toolkit that we
#   will rely on variables/function that exist within those
#   functions.
inherits: Boundary Grid MethodofLines

# Needed functions and #include's:
USES INCLUDE: Symmetry.h
USES INCLUDE: Boundary.h


# Needed Method of Lines function
CCTK_INT FUNCTION MoLRegisterEvolvedGroup(CCTK_INT IN EvolvedIndex, CCTK_INT IN RHSIndex)
REQUIRES FUNCTION MoLRegisterEvolvedGroup

# Needed Boundary Conditions function
CCTK_INT FUNCTION GetBoundarySpecification(CCTK_INT IN size, CCTK_INT OUT ARRAY nboundaryzones, CCTK_INT OUT ARRAY is_internal, CCTK_INT OUT ARRAY is_staggered, CCTK_INT OUT ARRAY shiftout)
USES FUNCTION GetBoundarySpecification

CCTK_INT FUNCTION SymmetryT

The CCL files connect NRPy registrations to Einstein Toolkit metadata:
variables, parameters, storage, and schedule entries.

# Step 9: Inspect Build Metadata and Source
### [Back to [top](#Table-of-Contents)]

`make.code.defn` lists the C files compiled into the thorn. The complete RHS
source is the required CFunction evidence: inspect the includes, scheduled
function name, parameter declarations, grid loop, and `psi_rhs` assignment.

In [14]:
make_code_defn_text = (THORN_DIR / "src" / "make.code.defn").read_text(
    encoding="utf-8", errors="replace"
)
rhs_source_text = (THORN_DIR / "src" / f"{THORN_NAME}_rhs_eval.c").read_text(
    encoding="utf-8", errors="replace"
)
print("--- src/make.code.defn ---")
print(make_code_defn_text)
print(f"--- src/{THORN_NAME}_rhs_eval.c ---")
print(rhs_source_text)

--- src/make.code.defn ---
# make.code.defn file for thorn ToyETLegacyNRPy

# Source files that need to be compiled:
SRCS = \
       ToyETLegacyNRPy_MoL_registration.c \
       ToyETLegacyNRPy_Symmetry_registration_oldCartGrid3D.c \
       ToyETLegacyNRPy_rhs_eval.c \
       ToyETLegacyNRPy_specify_Driver_BoundaryConditions.c \
       ToyETLegacyNRPy_zero_rhss.c

--- src/ToyETLegacyNRPy_rhs_eval.c ---
#include "cctk.h"
#include "cctk_Arguments.h"
#include "cctk_Parameters.h"

/**
 * Evaluate a toy right-hand side.
 */
void ToyETLegacyNRPy_rhs_eval(CCTK_ARGUMENTS) {
  DECLARE_CCTK_ARGUMENTS_ToyETLegacyNRPy_rhs_eval;
  DECLARE_CCTK_PARAMETERS;
  const CCTK_REAL *wave_speedptr = CCTK_ParameterGet("wave_speed", "ToyETLegacyNRPy", NULL); // ToyETLegacyNRPy::wave_speed
  const CCTK_REAL wave_speed = *wave_speedptr;
  const CCTK_REAL invdxx0 = 1.0 / CCTK_DELTA_SPACE(0);
  const CCTK_REAL invdxx1 = 1.0 / CCTK_DELTA_SPACE(1);
  const CCTK_REAL invdxx2 = 1.0 / CCTK_DELTA_SPACE(2);
#pragma omp pa

The complete RHS source is the generated C function promised by the
registration section. It is the concrete source artifact, not merely a
prototype.

# Validation Check
### [Back to [top](#Table-of-Contents)]

Trusted source: the NRPy CFunction registry and ETLegacy CCL/source writers
linked above. Newly checked output: the generated `ToyETLegacyNRPy` thorn in
this notebook run. A full external runtime test would require an Einstein
Toolkit checkout, so this validation checks generated content only.

In [15]:
validation_specs = [
    ("inventory", thorn_inventory == EXPECTED_THORN_INVENTORY, "exact file list"),
    ("interface implements", "implements: ToyETLegacyNRPy" in interface_text),
    ("interface inherits", "inherits: Boundary Grid MethodofLines" in interface_text),
    (
        "interface evol",
        all(item in interface_text for item in ["CCTK_REAL evol_variables", "psiGF"]),
    ),
    (
        "interface rhs",
        "psi_rhsGF" in interface_text,
    ),
    (
        "interface aux",
        all(item in interface_text for item in ["CCTK_REAL aux_variables", "energyGF"]),
    ),
    ("param wave_speed", "CCTK_REAL wave_speed" in param_text),
    ("param fd_order", "CCTK_INT fd_order" in param_text),
    ("schedule RHS", f"schedule {THORN_NAME}_rhs_eval in MoL_CalcRHS" in schedule_text),
    ("schedule reads", "READS: evol_variables(everywhere)" in schedule_text),
    ("schedule writes", "WRITES: evol_variables_rhs(interior)" in schedule_text),
    ("make RHS", f"{THORN_NAME}_rhs_eval.c" in make_code_defn_text),
    (
        "RHS source",
        all(
            item in rhs_source_text
            for item in [
                "DECLARE_CCTK_ARGUMENTS_ToyETLegacyNRPy_rhs_eval",
                "DECLARE_CCTK_PARAMETERS",
                "wave_speed",
                "psi_rhs",
                "energy",
                "psi",
            ]
        ),
    ),
]

for check in validation_specs:
    name, passed = check[0], bool(check[1])
    detail = check[2] if len(check) > 2 else "required substring"
    print_check(name, passed, detail)

failed = [check[0] for check in validation_specs if not check[1]]
if failed:
    raise RuntimeError("Validation failed: " + ", ".join(failed))
print("validation passed: ETLegacy thorn content matches the tutorial contract")

PASS | inventory | exact file list
PASS | interface implements | required substring
PASS | interface inherits | required substring
PASS | interface evol | required substring
PASS | interface rhs | required substring
PASS | interface aux | required substring
PASS | param wave_speed | required substring
PASS | param fd_order | required substring
PASS | schedule RHS | required substring
PASS | schedule reads | required substring
PASS | schedule writes | required substring
PASS | make RHS | required substring
PASS | RHS source | required substring
validation passed: ETLegacy thorn content matches the tutorial contract


The validation proves that the generated thorn contains the expected CCL
declarations, schedule metadata, build metadata, and complete RHS source.

# What Next?

A full Einstein Toolkit build requires copying the generated thorn into a
Cactus checkout. Replace `/path/to/Cactus` and `/path/to/repo` with local paths:

```bash
cd /path/to/Cactus
mkdir -p arrangements/NRPyTutorial
cp -R /path/to/repo/5-infrastructures/project/etlegacy_core_infrastructure/\
toy_etlegacy/ToyETLegacyNRPy arrangements/NRPyTutorial/
printf '%s\n' NRPyTutorial/ToyETLegacyNRPy > thornlists/nrpy_tutorial_etlegacy.th
./simfactory/bin/sim build -j2 --thornlist thornlists/nrpy_tutorial_etlegacy.th
```

Continue with:

- [Carpet WaveToy Thorns](carpet_wavetoy_thorns.ipynb)
- [Code-Writing Path Choice Guide](backend_choice_guide.ipynb)